In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc pandas qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Quantum Portfolio Optimizer: Isang Qiskit Function ng Global Data Quantum


> **Note:** Ang mga Qiskit Function ay isang eksperimental na feature na available lamang sa mga gumagamit ng IBM Quantum&reg; Premium Plan, Flex Plan, at On-Prem (sa pamamagitan ng IBM Quantum Platform API) Plan. Nasa preview release status ang mga ito at maaaring magbago.
# Pangkalahatang-ideya
Ang Quantum Portfolio Optimizer ay isang Qiskit Function na tumutugon sa dynamic portfolio optimization na problema — isang karaniwang problema sa pananalapi na naglalayong muling i-balance ang mga pana-panahong pamumuhunan sa iba't ibang asset upang mapakinabangan ang kita at mabawasan ang panganib. Sa pamamagitan ng paggamit ng pinakabagong mga teknik sa quantum optimization, pinapasimple ng function na ito ang proseso para makinabang ang mga gumagamit — kahit walang kaalaman sa quantum computing — mula sa mga kalamangan nito sa paghanap ng pinakamainam na mga trajectory ng pamumuhunan. Perpekto para sa mga portfolio manager, mananaliksik sa quantitative finance, at mga indibidwal na mamumuhunan, pinapahintulutan ng tool na ito ang back-testing ng mga trading strategy sa portfolio optimization.
# Paglalarawan ng function
Ginagamit ng Quantum Portfolio Optimizer function ang Variational Quantum Eigensolver (VQE) algorithm para malutas ng Quadratic Unconstrained Binary Optimization (QUBO) na problema, na tumutugon sa mga dynamic portfolio optimization na problema. Kailangan lamang ng mga gumagamit na ibigay ang datos ng presyo ng asset at tukuyin ang constraint ng pamumuhunan, pagkatapos ay pinapatakbo ng function ang proseso ng quantum optimization na nagbabalik ng isang set ng mga na-optimize na trajectory ng pamumuhunan.

Ang proseso ay binubuo ng apat na pangunahing yugto. Una, ang input data ay imi-map sa isang quantum-compatible na problema, itinatayo ang QUBO ng dynamic portfolio optimization na problema, at binabago ito sa isang quantum operator (Ising Hamiltonian). Susunod, ang input na problema at ang VQE algorithm ay ina-adapt para patakbuhin sa quantum hardware. Pagkatapos ay pinapatakbo ang VQE algorithm sa quantum hardware, at sa wakas, ang mga resulta ay pinoproseso upang maibigay ang pinakamainam na mga trajectory ng pamumuhunan. Kasama rin sa sistema ang isang noise-aware ([SQD](/guides/qiskit-addons-sqd)-based) na post-processing upang mapakinabangan ang kalidad ng output.

Ang Qiskit Function na ito ay batay sa [nailathala na manuskrito](https://arxiv.org/abs/2412.19150) ng Global Data Quantum.
![Visualization ng workflow ng function](../docs/images/guides/global-data-quantum-optimizer/function_workflow.svg)
# Input
Ang mga input argument ng function ay inilalarawan sa sumusunod na talahanayan. Ang datos ng mga asset at iba pang mga detalye ng problema ay dapat ibigay; bukod pa rito, maaaring isama ang mga setting ng VQE para i-customize ang proseso ng optimization.
| **Pangalan**               | **Uri**           | **Paglalarawan**                                          | **Kinakailangan** | **Default**                          | **Halimbawa**                              |
|------------------------|--------------------|----------------------------------------------------------|--------------|--------------------------------------|------------------------------------------|
| assets                 | json               | Dictionary na may mga presyo ng asset                         | Oo          | -                                    | -                                        |
| qubo_settings          | json               | Mga setting ng QUBO                                     | Oo          | -                                    | Tingnan ang mga halimbawa sa talahanayan sa ibaba     |
| ansatz_settings        | json               | Mga setting ng ansatz                                   | Hindi           | `None`                                | Tingnan ang mga halimbawa sa talahanayan sa ibaba.     |
| optimizer_settings     | json               | Mga setting ng optimizer                                | Hindi           | `None`                                | Tingnan ang mga halimbawa sa talahanayan sa ibaba.     |
| backend                | str                | Pangalan ng QPU backend                                     | Hindi           |  -    | `"ibm_torino"`                           |
| previous_session_id    | list of str        | Listahan ng mga session ID para makuha ang datos mula sa mga nakaraang run[(*)](#1-note) | Hindi           | Walang laman na listahan                           | `["session_id_1", "session_id_2"]`      |
| apply_postprocess      | bool               | Mag-apply ng noise-aware SQD post-processing                      | Hindi           | `True`                                | `True`                                   |
| tags                   | list of strings    | Listahan ng mga tag para makilala ang eksperimento                  | Hindi           | Walang laman na listahan                           | `["optimization", "quantum_computing"]`  |
<span id="1-note"></span>
*Para i-resume ang isang eksekusyon o makuha ang mga job na naproseso sa isa o higit pang mga nakaraang session, ang listahan ng mga session ID ay dapat ipasa sa parameter na `previous_session_id`. Ito ay lalo na kapaki-pakinabang sa mga kaso kung saan ang isang optimization task ay hindi nakumpleto dahil sa anumang error sa proseso, at kailangang tapusin ang eksekusyon. Para magawa ito, dapat mong ibigay ang parehong mga argumento na ginamit sa paunang eksekusyon, kasama ang listahan ng `previous_session_id` tulad ng inilarawan.*
> **Caution:** Ang pag-load ng datos para sa mga nakaraang session (para sa pag-resume ng optimization) ay maaaring tumagal ng hanggang isang oras.
## `assets`
Ang datos ay dapat nakastruktura bilang isang JSON object na nag-iimbak ng impormasyon tungkol sa mga closing price ng mga financial asset sa mga partikular na petsa. Ang format ay ang sumusunod:

- Pangunahing key (string): Ang pangalan o ticker symbol ng financial asset (halimbawa, "8801.T").
- Pangalawang key (string): Ang petsa sa format na YYYY-MM-DD.
- Value (number): Ang closing price ng asset sa tinukoy na petsa. Maaaring ipasok ang mga presyo nang normalized o hindi-normalized.

*Pansinin na ang lahat ng dictionary ay dapat may parehong pangalawang key (mga petsa). Kung ang isang partikular na asset ay walang petsa na mayroon ang iba, ang datos ay dapat punan upang masiguro ang konsistensya. Halimbawa, maaari itong gawin sa pamamagitan ng paggamit ng huling na-track na closing price ng asset na iyon.*
### Halimbawa
``` py
{
    "8801.T": {
        "2023-01-01": 2374.0,
        "2023-01-02": 2374.0,
        "2023-01-03": 2374.0,
        "2023-01-04": 2356.5,
        ...
    },
    "AAPL": {
        "2023-01-01": 145.2,
        "2023-01-02": 146.5,
        "2023-01-03": 147.3,
        "2023-01-04": 148.1,
        ...
    },
    ...
}
```

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access function
dpo_solver = catalog.load("global-data-quantum/quantum-portfolio-optimizer")

> **Note:** Ang datos ng asset ay dapat maglaman, kahit man lang, ng mga closing price sa ``(nt+1) * dt`` (tingnan ang seksyon ng input na [`qubo_settings`](#qubo_settings)) na mga time stamp (halimbawa, mga araw).
## `qubo_settings`
Inilalarawan ng susunod na talahanayan ang mga key ng dictionary na `qubo_settings`. Itayo ang dictionary sa pamamagitan ng pagtukoy ng bilang ng mga time step na `nt`, ng bilang ng resolution qubit na `nq`, at ng `max_investment` — o baguhin ang iba pang default na halaga.
| Pangalan                | Uri    | Paglalarawan                                                                 | Kinakailangan | Default | Halimbawa |
|---------------------|---------|-----------------------------------------------------------------------------|----------|---------|---------|
| nt                  | int     | Bilang ng mga time step                                                         | Oo      | -       | 4       |
| nq                  | int     | Bilang ng mga resolution qubit                                                                | Oo      | -       | 4       |
| max_investment      | float   | Pinakamataas na bilang ng mga invested currency unit sa lahat ng asset                           | Oo      | -       | 10      |
| dt*                  | int     | Time window na isinasaalang-alang sa bawat time step. Ang unit ay tumutugma sa mga time interval sa pagitan ng mga key sa datos ng asset                                                 | Hindi       | 30      | -       |
| risk_aversion       | float     | Coefficient ng risk aversion                                   | Hindi       | 1000    | -       |
| transaction_fee     | float     | Coefficient ng transaction fee                                                 | Hindi       | 0.01   | -       |
| restriction_coeff   | float   | Lagrange multiplier na ginagamit upang ipatupad ang constraint ng problema sa loob ng QUBO formulation | Hindi       | 1       | -       |
## `ansatz_settings`
Para baguhin ang mga default na opsyon, lumikha ng dictionary para sa parameter na `ansatz_settings` gamit ang mga sumusunod na key. Bilang default, ang ansatz ay nakatakda sa `"real_amplitudes"`, at ang parehong mga karagdagang opsyon (tingnan ang sumusunod na talahanayan) ay nakatakda sa `False`.
| Pangalan                  | Uri     | Paglalarawan                                                                 | Kinakailangan | Default |
|-----------------------|----------|-----------------------------------------------------------------------------|----------|---------|
| ansatz[*](#single-star)                | str      | Ansatz na gagamitin                                             | Hindi      | `"real_amplitudes"` |
| multiple_passmanager[**](#double-star)  | bool     |Nagpapagana ng multiple passmanager subroutine (hindi available para sa Tailored ansatz) | Hindi       | `False`   |
| dd_enable   | bool     | Nagdaragdag ng dynamical decoupling                                    | Hindi       | `False`   |

<span id="single-star"></span>
\* Mga available na ansatz
- `real_amplitudes`
- `cyclic`
- `optimized_real_amplitudes`
- `tailored` (Para lamang sa `ibm_torino` backend, 7 asset, 4 time step, at 4 resolution qubit)

<span id="double-star"></span>
** Kung ang ``multiple_passmanager`` ay nakatakda sa ``False``, ginagamit ng function ang default na Qiskit pass manager na may `optimization_level=3`. Kung nakatakda sa ``True``, inihahambing ng ``multiple_passmanager`` subroutine ang tatlong pass manager: ang nakaraang default na Qiskit pass manager, isang pass manager na nagma-map ng mga qubit sa QPU first neighbors chain, at ang mga serbisyo ng [AI transpiler](/guides/ai-transpiler-passes). Pagkatapos, ang pass manager na may tinatantyang mas mababang cumulative error ay napipili.
## `optimizer_settings`
Ang parameter na ito ay isang dictionary na may ilang nababagong opsyon ng proseso ng pag-optimize.
| Pangalan               | Uri   | Paglalarawan                                     | Kinakailangan | Default               |
|--------------------|--------|-------------------------------------------------|----------|-----------------------|
| primitive_options  | json   | Mga setting ng primitive               | Hindi      | -                     |
| optimizer          | str    | Napiling classical optimizer                            | Hindi       | `"differential_evolution"` |
| optimizer_options  | json   | Konfigurasyong ng optimizer                  | Hindi       | -                     |
> **Note:** Sa kasalukuyan, ang tanging available na opsyon ng optimizer ay ang `"differential_evolution"`.

Sa ilalim ng mga key na `primitive_options` at `optimizer_options`, nagtatakda tayo ng mga dictionary na may mga sumusunod na parameter:
### `primitive_options`
| Pangalan              | Uri   | Paglalarawan                                | Kinakailangan | Default                    | Halimbawa                    |
|-------------------|--------|--------------------------------------------|----------|----------------------------|----------------------------|
| sampler_shots     | int    | Bilang ng mga shot ng Sampler.            | Hindi       | 100000                     | -                          |
| estimator_shots   | int    | Bilang ng mga shot ng Estimator.          | Hindi       | 25000                      | -                          |
| estimator_precision | float | Nais na katumpakan ng expected value. Kung tinukoy, ang katumpakan ay gagamitin sa halip na `estimator_shots`. | Hindi | `None` | 0.015625 · (1 / sqrt(4096)) |
| max_time          | int o str | Pinakamataas na halaga ng oras na maaaring manatiling bukas ang isang runtime session bago puwersa itong isara. Maaaring ibigay sa segundos (int) o bilang string, tulad ng `"2h 30m 40s"`. Dapat na mas mababa sa system-imposed na maximum. | Hindi | `None` | `"1h 15m"`                |
### `optimizer_options`
| Pangalan              | Uri     | Paglalarawan                              | Kinakailangan | Default       |
|-------------------|----------|------------------------------------------|----------|---------------|
| num_generations   | int      | Bilang ng mga henerasyon                    | Hindi       | 20            |
| population_size   | int      | Laki ng populasyon                    | Hindi       | 20            |
| mutation_range    | list   | Pinakamataas at pinakamababa na mutation factor              | Hindi       | [0, 0.25]     |
| recombination     | float      | Recombination factor                     | Hindi       | 0.4           |
| max_parallel_jobs | int      | Pinakamataas na bilang ng mga QPU job na isinasagawa nang sabay-sabay  | Hindi       | 3             |
| max_batchsize | int      | Pinakamataas na laki ng batch  | Hindi       | 200     |
> **Note:** - Ang bilang ng mga henerasyong sinusuri ng differential evolution ay `num_generations` + 1 dahil kasama ang paunang populasyon.
> - Ang kabuuang bilang ng mga circuit ay kinakalkula bilang `(num_generations + 1) * population_size`.
> - Ang paggamit ng mas malaking laki ng populasyon at mas maraming henerasyon ay karaniwang nagpapabuti ng kalidad ng mga resulta ng optimization. Gayunpaman, hindi inirerekomenda na lumampas sa laki ng populasyong 120 at bilang ng mga henerasyong higit sa 20 (halimbawa, ``120 * 21 = 2520`` kabuuang circuit), dahil magagawa nitong labis na maramihang circuit, na maaaring mahal sa computationally at matagal na prosesahin.
> 
> - Pinapahintulutan ng function na i-resume ang nakaraang optimization, at lagi itong posible na dagdagan ang bilang ng mga henerasyon (sa pamamagitan ng pagbibigay ng parehong input maliban sa `previous_session_id` at mas mataas na ``num_generations``).
> **Note:** Tiyaking sumusunod sa mga limitasyon ng Qiskit Runtime job.
> 
> - Sampler: `sampler_shots <= 10_000_000`.
> - Estimator: `max_batchsize * estimator_shots * observable_size <= 10_000_000` (para sa function na ito, lahat ng term ng observable ay nagko-commute, kaya `observable_size=1`).
> 
> Tingnan ang gabay na [Mga limitasyon ng Job](/guides/job-limits) para sa karagdagang impormasyon.
# Output
Ang function ay nagbabalik ng dalawang dictionary: ang dictionary na `"result"`, na naglalaman ng pinakamahusay na mga resulta ng optimization, kasama ang pinakamainam na solusyon at ang pinakamababang objective cost nito; at ang `"metadata"`, na may datos mula sa lahat ng resulta na nakuha sa panahon ng proseso ng optimization, kasama ang kani-kanilang mga sukatan.

Ang unang dictionary ay nakatuon sa pinakamahusay na solusyon, habang ang pangalawa ay nagbibigay ng detalyadong impormasyon tungkol sa lahat ng solusyon, kasama ang mga objective cost at iba pang mga kaugnay na sukatan.

## Mga output dictionary:
| **Pangalan**    | **Uri**                     | **Paglalarawan**                                                                 | **Halimbawa**  |
|-------------|------------------------------|---------------------------------------------------------------------------------|--------------|
| result      | dict[str, dict[str, float]]  | Naglalaman ng estratehiya ng pamumuhunan sa paglipas ng panahon, kung saan ang bawat time-stamp ay naka-map sa mga investment weight na espesipiko sa asset (ang bawat weight ay ang halaga ng pamumuhunan na normalized sa kabuuang halaga ng pamumuhunan). | `{'time_1': {'asset_1': 0.2, 'asset_2': 0.3, ...}, ...}` |
| metadata    | dict[str, Any]               | Datos na nabuo sa panahon ng pagsusuri, kasama ang mga solusyon, gastos, at sukatan.    | Tingnan ang mga halimbawa sa ibaba |
### Paglalarawan ng dictionary na `metadata`
| **Pangalan**                 | **Uri**              | **Paglalarawan**                                                                                     | **Halimbawa**                  |
|--------------------------|-----------------------|-----------------------------------------------------------------------------------------------------|------------------------------|
| session_id               | str                   | Natatanging identifier para sa IBM Quantum session.                          | `"d0h30qjvpqf00084fgw0"`        |
| all_samples_metrics | dict                 | Dictionary na naglalaman ng iba't ibang sukatan para sa bawat postprocessed sample, tulad ng mga gastos o mga constraint. | Tingnan ang paglalarawan sa ibaba        |
| sampler_counts           | dict[str, int]        | Dictionary kung saan ang mga key ay bitstring na representasyon ng mga sample na solusyon at ang mga value ay ang kanilang bilang. | `{"101010": 3, "111000": 1}` |
| asset_order              | list[str]             | Listahan na may kaukulang investment order ng mga asset sa bawat time step sa loob ng mga estratehiya sa pamumuhunan. | `["Asset_0", "Asset_1", "Asset_3"]`     |
| QUBO                     | list[list[float]]     | QUBO matrix ng problema.                                                                         | `[[-6.96e-01, 5.81e-01, -1.26e-02, 0.00e+00], ...]`     |
| resource_summary           | dict[str, dict[str, float]] | Buod ng mga oras ng paggamit ng CPU at QPU (sa segundos) sa iba't ibang yugto ng proseso.                | `{'RUNNING: EXECUTING_QPU': {'CPU_TIME': 412.84, 'QPU_TIME': 87.22}, ...}` |
### Paglalarawan ng dictionary na `all_samples_metrics`
| **Pangalan**                | **Uri**       | **Paglalarawan**                                                                                                  | **Halimbawa**                  |
|-------------------------|----------------|------------------------------------------------------------------------------------------------------------------|------------------------------|
| investment_trajectories | list[list]     | Mga estratehiya sa pamumuhunan na nagmula sa mga na-decode na quantum state. | `[[1, 2, 2], [1, 2, 1]]`     |

| counts                  | list[int]      | Bilang ng beses na na-sample ang bawat investment trajectory. Ang index ay tumutugma sa `investment_trajectories`.                | `[5, 3]`                     |
| objective_costs         | list[float]    | Halaga ng objective function para sa bawat investment trajectory, nakaayos mula sa pinakamababa hanggang pinakamataas.                 | `[0.98, 1.25]`               |
| sharpe_ratios           | list[float]    | Risk-adjusted na pagganap (Sharpe ratio) para sa bawat investment trajectory. Nakahanay ayon sa index.                      | `[1.1, 0.7]`                 |
| returns                 | list[float]    | Inaasahang kita para sa bawat investment trajectory. Nakahanay ayon sa index.                                               | `[0.15, 0.10]`               |
| rest_breaches           | list[float]    | Pinakamataas na paglabag sa constraint sa loob ng bawat investment trajectory. Nakahanay ayon sa index.                               | `[0.0, 0.25]`                |
| transaction_costs       | list[float]    | Tinatantyang transaction cost na nauugnay sa bawat investment trajectory. Nakahanay ayon sa index.                        | `[0.01, 0.02]`               |
# Magsimula
Mag-authenticate gamit ang iyong [API key](https://quantum.cloud.ibm.com/) at piliin ang Qiskit Function tulad ng sumusunod. (Ipinapalagay ng snippet na ito na nai-save mo na ang iyong [account](/guides/functions#install-qiskit-functions-catalog-client) sa iyong lokal na kapaligiran.)

In [ ]:
import os
import glob
import pandas as pd


def read_and_join_csv(file_pattern):
    """
    Reads multiple CSV files matching the file pattern and combines them into a single DataFrame.

    Parameters:
    file_pattern (str): The pattern to match CSV files.

    Returns:
    pd.DataFrame: Combined DataFrame with data from all CSV files.
    """
    # Find all files matching the pattern
    csv_files = glob.glob(file_pattern)
    # Get the base file names without the .csv extension
    file_names = [os.path.basename(f).replace(".csv", "") for f in csv_files]
    # Read each CSV file into a DataFrame and set the first column as the index
    df_list = [pd.read_csv(f).set_index("Unnamed: 0") for f in csv_files]

    # Rename columns in each DataFrame to the base file names
    for df, name in zip(df_list, file_names):
        df.columns = [name]

    # Combine all DataFrames into one by merging them side by side
    combined_df = pd.concat(df_list, axis=1)
    return combined_df


file_pattern = "route/to/folder/with/assets/data/*.csv"
assets = read_and_join_csv(file_pattern).to_dict()

## Halimbawa: Dynamic portfolio optimization na may pitong asset
Ipinapakita ng halimbawang ito kung paano isagawa ang dynamic portfolio optimization (DPO) function at i-adjust ang mga setting nito para sa pinakamainam na pagganap. Kasama nito ang mga detalyadong hakbang para i-fine-tune ang mga parameter upang makamit ang mga nais na resulta.

Ang kasong ito ay kinabibilangan ng pitong asset, apat na time step, at apat na resolution qubit, na nagresulta sa kabuuang pangangailangan na 112 qubit.
### 1. Basahin ang mga asset na kasama sa portfolio.
Kung ang lahat ng asset sa portfolio ay nakaimbak sa isang folder sa isang partikular na path, maaari mo itong i-load sa isang `pandas.DataFrame` at i-convert ito sa isang object na may format na `dict` gamit ang sumusunod na function.

In [ ]:
qubo_settings = {
    "nt": 4,
    "nq": 4,
    "dt": 30,
    "max_investment": 25,
    "risk_aversion": 1000.0,
    "transaction_fee": 0.01,
    "restriction_coeff": 1.0,
}

Para sa halimbawang ito, ginamit namin ang mga asset na [8801.T](https://finance.yahoo.com/quote/8801.T), [CLF](https://finance.yahoo.com/quote/CLF), [GBPJPY](https://finance.yahoo.com/quote/GBPJPY), [ITX.MC](https://finance.yahoo.com/quote/ITX.MC), [META](https://finance.yahoo.com/quote/META), [TMBMKDE-10Y](https://finance.yahoo.com/quote/TMBMKDE-10Y), at [XS2239553048](https://finance.yahoo.com/quote/XS2239553048). Inilalarawan ng sumusunod na figure ang datos na ginamit sa halimbawang ito, na nagpapakita ng ebolusyon ng pang-araw-araw na closing price ng mga asset mula 1 Enero hanggang 1 Setyembre 2023.

Sa halimbawang ito, para matiyak ang pagkakatugma sa iba't ibang petsa, pinuno namin ang mga araw na hindi nagtatrabaho ng closing price mula sa nakaraang available na petsa. Inilalapat namin ang hakbang na ito dahil ang mga napiling asset ay nagmumula sa iba't ibang merkado na may magkakaibang araw ng pangangalakal, na ginagawa itong mahalaga na i-standardize ang dataset para sa konsistensya.
![Visualization ng makasaysayang datos ng mga asset](../docs/images/guides/global-data-quantum-optimizer/assets.avif)

### 2. Tukuyin ang problema.

Tukuyin ang mga detalye ng problema sa pamamagitan ng pag-configure ng mga parameter sa dictionary na `qubo_settings`.

In [ ]:
optimizer_settings = {
    "de_optimizer_settings": {
        "num_generations": 20,
        "population_size": 90,
        "recombination": 0.4,
        "max_parallel_jobs": 5,
        "max_batchsize": 4,
        "mutation_range": [0.0, 0.25],
    },
    "optimizer": "differential_evolution",
    "primitive_settings": {
        "estimator_shots": 25_000,
        "estimator_precision": None,
        "sampler_shots": 100_000,
    },
}

### 3. Tukuyin ang mga setting ng optimizer at ansatz. (Opsyonal)
Opsyonal na tukuyin ang mga tiyak na pangangailangan para sa proseso ng optimization, kasama ang pagpili ng optimizer at mga parameter nito, pati na rin ang pagtukoy ng primitive at mga konfigurasyong nito.

Para sa Tailored Ansatz, ang napiling laki ng populasyon ay batay sa mga nakaraang eksperimento na nagpapakita na ang halagang ito ay nagbubunga ng stable at mahusay na optimization.

Sa kaso ng Real Amplitudes Ansatz, maaari kang sumunod sa linear na relasyon sa pagitan ng ``population_size`` at ng bilang ng mga qubit sa circuit. Bilang isang tinatayang rule of thumb, inirerekomenda na gumamit ng **minimum** na ``population_size ~ 0.8 * n_qubits`` para sa `real_amplitudes` ansatz.

Inaasahang magkakaroon ang Optimized Real Amplitudes ng mas mahusay na optimization performance kaysa sa Real Amplitudes ansatz. Gayunpaman, ang bilang ng mga variable na ino-optimize sa ansatz na ito ay tumataas nang mas mabilis kaysa sa kaso ng Real Amplitudes (tingnan ang [manuskrito](https://arxiv.org/pdf/2412.19150)). Kaya, para sa malalaking problema, ang Optimized Real Amplitudes ay nangangailangan ng mas maraming circuit execution. Ang Optimized Real Amplitudes ay malamang na kapaki-pakinabang para sa mga problemang nangangailangan ng hanggang 100 qubit, ngunit inirerekomenda na maging maingat kapag nagtatakda ng mga parameter na ``population_size``. Bilang halimbawa ng scale-up na ito sa ``population_size``, ang nakaraang talahanayan ay nagpapakita na para sa isang 84-qubit na problema, ang Optimize Real Amplitudes ay nangangailangan ng 120 na ``population_size``, habang para sa isang 56-qubit na problema, sapat na ang ``population_size`` na 40.

In [ ]:
ansatz_settings = {
    "ansatz": "tailored",
    "multiple_passmanager": False,
}

Posible rin na pumili ng isang tiyak na ansatz. Gumagamit ang sumusunod ng ansatz na ``'Tailored'``.

In [ ]:
dpo_job = dpo_solver.run(
    assets=assets,
    qubo_settings=qubo_settings,
    optimizer_settings=optimizer_settings,
    ansatz_settings=ansatz_settings,
    backend_name="<backend name>",
    previous_session_id=[],
    apply_postprocess=True,
)

### 4. Patakbuhin ang problema.

In [ ]:
# Get the results of the job
dpo_result = dpo_job.result()

# Show the solution strategy
dpo_result["result"]

{'time_step_0': {'8801.T': 0.11764705882352941,
  'ITX.MC': 0.20588235294117646,
  'META': 0.38235294117647056,
  'GBPJPY=X': 0.058823529411764705,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.058823529411764705,
  'XS2239553048': 0.17647058823529413},
 'time_step_1': {'8801.T': 0.11428571428571428,
  'ITX.MC': 0.14285714285714285,
  'META': 0.2,
  'GBPJPY=X': 0.02857142857142857,
  'TMBMKDE-10Y': 0.42857142857142855,
  'CLF': 0.0,
  'XS2239553048': 0.08571428571428572},
 'time_step_2': {'8801.T': 0.0,
  'ITX.MC': 0.09375,
  'META': 0.3125,
  'GBPJPY=X': 0.34375,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.25},
 'time_step_3': {'8801.T': 0.3939393939393939,
  'ITX.MC': 0.09090909090909091,
  'META': 0.12121212121212122,
  'GBPJPY=X': 0.18181818181818182,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.21212121212121213}}

### 5. Kunin ang mga resulta.
Tulad ng nabanggit sa seksyon ng [Output](#output), ang function ay nagbabalik ng dictionary na may mga investment trajectory na nakaayos mula sa pinakamababa hanggang pinakamataas ayon sa halaga ng kanilang objective function. Ang set ng mga resulta na ito ay nagbibigay-daan sa pagkilala ng trajectory na may pinakamababang gastos at ang kaukulang mga pagsusuri ng pamumuhunan. Bukod pa rito, nagbibigay ito para sa pagsusuri ng iba't ibang trajectory, na nagpapadali sa pagpili ng mga pinaka-angkop sa mga tiyak na pangangailangan o layunin. Tinitiyak ng flexibility na ito na ang mga pagpipilian ay maaaring iayon upang tumugon sa iba't ibang kagustuhan o sitwasyon.
Simulan sa pamamagitan ng pagpapakita ng resultang estratehiya na nakamit ang pinakamababang objective cost na natagpuan sa proseso.

In [ ]:
# Convert metadata to a DataFrame
df = pd.DataFrame(dpo_result["metadata"]["all_samples_metrics"])

# Find the minimum objective cost
min_cost = df["objective_costs"].min()
print(f"Minimum Objective Cost Found: {min_cost:.2f}")

# Extract the row with the lowest cost
best_row = df[df["objective_costs"] == min_cost].iloc[0]

# Display the results associated with the best solution
print("Best Solution:")
print(f"  - Restriction Deviation: {best_row['rest_breaches']}%")
print(f"  - Sharpe Ratio: {best_row['sharpe_ratios']:.2f}")
print(f"  - Return: {best_row['returns']}")

Minimum Objective Cost Found: -3.78
Best Solution:
  - Restriction Deviation: 40.0
  - Sharpe Ratio: 24.82
  - Return: 0.46


### 6. Performance analysis

Last, analyze the performance of your optimization application. Specifically, compare your results, obtained in the previous example, against a random baseline to assess the effectiveness of our approach. If the quantum algorithm demonstrably and consistently produces results with lower cost values, it indicates an effective optimization process.

The figure presents the probability distributions of the objective costs. To generate these distributions, take the list of objective costs from the function result and count the occurrences of each cost value (values rounded to the second decimal place). Then, update the count column accordingly by joining counts of identical rounded values. Note that, for better visual comparison, the occurrence counts have been normalized so that each distribution is displayed between 0 and 1.

![Visualization of the solution of the optimization](../docs/images/guides/global-data-quantum-optimizer/cost_distribution.svg)

As shown in the figure (blue solid line), the cost distribution for our Variational Quantum Eigensolver (post-processed with SQD) approach is sharply concentrated at lower objective cost values, indicating good optimization performance. In contrast, the noisy baseline exhibits a broader distribution, centered around higher cost values. The gray dashed vertical line represents the mean value of the random distribution, further highlighting the consistency of the function in returning optimized investment strategies. For an additional comparison, the black dashed line in the figure corresponds to the solution obtained with the Gurobi optimizer (free version). All these results are further explored in the benchmarks below for the "Mixed Assets" example evaluated with the "Tailored" ansatz.

# Benchmarks

This function was tested under different configurations of resolution qubits, ansatz circuits, and groupings of assets from various sectors: a mix of different assets (Set 1), oil derivatives (Set 2), and IBEX35 (Set 3). See more details in the following table.

| Set       | Date       | Assets                                                                 |
|-----------|------------|------------------------------------------------------------------------|
| **Set 1** | 01/01/2023 | 8801.T, CL=F, GBPJPY=X, ITX.MC, META, TMBMKDE-10Y, XS2239553048         |
| **Set 2** | 01/06/2023 | CL=F, BZ=F, HO=F, NG=F, XOM, RB=F, 2222.SR                               |
| **Set 3** | 01/11/2022 | ACS.MC, ITX.MC, FER.MC, ELE.MC, SCYR.MC, AENA.MC, AMS.MC                |

Two key metrics were used to evaluate solution quality.
1. The objective cost, which measures optimization efficiency by comparing the cost function value from each experiment with results from Gurobi (free version).
2. The Sharpe ratio, which captures the risk-adjusted return of each portfolio, offering insight into the financial performance of the solutions.

Together, these metrics benchmark both computational and financial aspects of the quantum-generated portfolios.


| Example             | Qubits | Ansatz                  | Depth | Runtime Usage (s) | Total usage (s) | Objective cost | Sharpe | Gurobi objective cost | Gurobi Sharpe |
|-------------------------------|--------|-------------------------------|--------|-------------------|------------------|----------------|--------|------------------------|----------------|
| Mixed Assets (Set 1, 4 time steps, 4-bit)   | 112    | Tailored                      | 83     | 12735             | 13095            | -3.78          | 24.82  | -4.25                  | 24.71          |
| Mixed Assets (Set 1,4 time steps, 4 time steps, 4-bit)   | 112    | Real Amplitudes               | 359    | 11739             | 11903            | -3.39          | 23.64  | -4.25                  | 24.71          |
| Oil Derivatives (Set 2, 4 time steps, 3-bit)| 84     | Optimized Real Amplitudes     | 78     | 6180              | 6350             | -3.73          | 19.13  | -4.19                  | 21.71          |
| IBEX35 (Set 3, 4 time steps, 2-bit)         | 56     | Optimized Real Amplitudes     | 96     | 3314              | 3523             | -3.67          | 14.48  | -4.11                  | 16.44          |



Results show that the quantum optimizer, with problem-specific ansatzes, effectively identifies efficient investment strategies across various portfolio types.

Below we detail both the population size and the number of generations specified in the `optimizer_options` dictionary. All other parameters were set to their default values.

| **Example**                | ``population_size`` | ``num_generations``   |
|----------------------------|----------------------|----------------------|
| Mixed Assets Portfolio     | 90                   | 20                   |
| Mixed Assets Portfolio     | 92                   | 20                   |
| Oil Derivatives Portfolio  | 120                  | 20                   |
| IBEX35 Portfolio           | 40                   | 20                   |

The number of generations was set to 20, as this value was found to be sufficient to reach convergence. Additionally, the default values for the optimizer's internal parameters were left unchanged, as they consistently provided good performance and are generally recommended by the literature and implementation guidelines.

# Get support
If you need help, you can send an email to qpo.support@globaldataquantum.com. In your message, provide the function job ID.

## Next steps

<Admonition type="note" title="Recommendations">
*   Read [the associated research paper](https://arxiv.org/pdf/2412.19150).
*   Request access to the function by filling in [this form](https://www.globaldataquantum.com/en/quantum-portfolio-optimizer/#form).
*   Try the [Dynamic Portfolio Optimization](/docs/tutorials/global-data-quantum-optimizer) tutorial.

</Admonition>